# LensIQ: synthetic data seed\n\nPopulates demo tables for the **LensIQ** AppKit app demo.\nTables are written under `${catalog}.${schema}` with sensible defaults\n(`reggie_pierce_aws_catalog.lens_iq`). Re-run idempotently: the script drops and recreates\neach table before inserting fresh data, then applies table + column COMMENTs so the\nGenie space picks up rich descriptions.\n\nBaseline (server-side analytics):\n\n- `stores`            (one row per location)\n- `devices`           (temperature sensors, status)\n- `device_readings`   (historical temperature / humidity samples)\n- `camera_status`     (online/offline samples per camera, last 24h)\n- `detections`        (CV bounding boxes: pizza, vehicle, person, truck, package)\n- `license_plates`    (drive-through plate captures)\n- `inventory`         (pizza stock + truck parking, last 12h)\n- `alerts`            (Jolt rule events, last 7 days)\n\nUC mirrors of the LensIQ app's Lakebase write-backs so Genie can answer\nbusiness questions about live demo data the moment those pages are exercised:\n\n- `guest_counts`      (per-zone person counts from the Guests page)\n- `plate_reads`       (per-OCR plate reads from the Plates page)\n- `fog_observations`  (per-tick lens condition from the Camera Health page)\n- `spill_cycles`      (per-cycle spill -> cone response time from the Spills page)\n- `face_matches`      (per-tick banned/vip/staff face matches from the Facial Recognition page)\n\nThe mirror tables are seeded with a few days of synthetic data so the Genie\nspace returns useful answers even before any booth demo has been run. The\ndeployed app dual-writes to them via `analytics.query` so live demo activity\nbecomes Genie-readable within ~2 seconds.

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_aws_catalog")
dbutils.widgets.text("schema", "lens_iq")
dbutils.widgets.text("volume", "frames")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
FQN = f"{CATALOG}.{SCHEMA}"

In [ ]:
import logging
import random
from datetime import datetime, timedelta, timezone

from pyspark.sql import Row

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("seed_data")

_RNG = random.Random(42)
_NOW = datetime.now(timezone.utc).replace(microsecond=0)

_STORE_DEFS: list[tuple[str, str, str, float, float]] = [
    ("S-ATL-001", "Store #1247 - Atlanta",        "Atlanta, GA",      33.7490, -84.3880),
    ("S-ATL-002", "Store #1248 - Atlanta North",   "Atlanta, GA",      33.9526, -84.5499),
    ("S-DAL-001", "Store #2145 - Dallas",          "Dallas, TX",       32.7767, -96.7970),
    ("S-HOU-001", "Store #2389 - Houston",         "Houston, TX",      29.7604, -95.3698),
    ("S-TAM-001", "Store #3421 - Tampa",           "Tampa, FL",        27.9506, -82.4572),
    ("S-TAM-002", "Store #3422 - Tampa Bay",       "Tampa, FL",        27.7676, -82.6403),
    ("S-NAS-001", "Store #4108 - Nashville",       "Nashville, TN",    36.1627, -86.7816),
    ("S-CHA-001", "Store #4215 - Charlotte",       "Charlotte, NC",    35.2271, -80.8431),
]

_LABELS = ["vehicle", "person", "truck", "package", "pizza"]
_STATES = ["GA", "FL", "TX", "AL", "SC", "NC", "TN", "MS"]
_STATE_WEIGHTS = [0.28, 0.24, 0.16, 0.11, 0.08, 0.06, 0.04, 0.03]

In [ ]:
# Assume the catalog already exists (most workspaces with Default Storage
# disallow ad-hoc CREATE CATALOG). We only ensure the schema + volumes.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQN}.{VOLUME}")
# `frames_inbox` is the drop point for the continuous detection pipeline:
# the simulator (or a Zerobus producer) writes raw images here, Auto Loader
# picks them up, and the pipeline dedupes + runs YOLO.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQN}.frames_inbox")
LOG.info("Schema/volumes ready: %s.%s (volumes=%s, frames_inbox)", CATALOG, SCHEMA, VOLUME)

## Stores

In [ ]:
store_rows = [
    Row(id=sid, name=name, location=loc, lat=lat, lng=lng)
    for sid, name, loc, lat, lng in _STORE_DEFS
]
(spark.createDataFrame(store_rows)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.stores"))
LOG.info("Wrote %d stores", len(store_rows))

## Devices + readings

One temperature device per store, with a status drawn from a triangular
distribution around its current temperature. Readings are written hourly for
the last 7 days.

In [ ]:
def _classify(temp: float) -> str:
    if temp > 90.0:
        return "critical"
    if temp > 80.0:
        return "warning"
    return "normal"


def _build_devices() -> tuple[list[Row], list[Row]]:
    device_rows: list[Row] = []
    reading_rows: list[Row] = []
    for sid, name, loc, _lat, _lng in _STORE_DEFS:
        device_id = sid.replace("S-", "RT-") + "-T1"
        base = _RNG.uniform(70.0, 90.0)
        current = round(base + _RNG.uniform(-2.5, 5.0), 1)
        device_rows.append(Row(
            id=device_id,
            name=name,
            location=loc,
            current_temp=current,
            status=_classify(current),
            last_update=_NOW - timedelta(seconds=_RNG.randint(30, 600)),
        ))
        for hours_ago in range(0, 24 * 7):
            ts = _NOW - timedelta(hours=hours_ago)
            wave = 8.0 * (1.0 + (0.5 * (hours_ago % 24) / 24.0))
            temp = round(base + wave * (_RNG.random() - 0.5), 1)
            humidity = round(45.0 + _RNG.uniform(0, 25), 1)
            reading_rows.append(Row(
                device_id=device_id,
                ts=ts,
                temperature=temp,
                humidity=humidity,
                status=_classify(temp),
            ))
    return device_rows, reading_rows


_devices, _readings = _build_devices()
(spark.createDataFrame(_devices)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.devices"))
(spark.createDataFrame(_readings)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.device_readings"))
LOG.info("Wrote %d devices, %d readings", len(_devices), len(_readings))

## Camera status

Per-hour online flag for ~20 cameras across all stores. Used for the
"Online Cameras Frequency" chart on the overview.

In [ ]:
def _build_camera_status() -> list[Row]:
    cameras_per_store = 3
    rows: list[Row] = []
    for sid, *_ in _STORE_DEFS:
        for cam in range(cameras_per_store):
            cam_id = f"{sid}-CAM-{cam+1:02d}"
            for hours_ago in range(0, 24):
                ts = _NOW - timedelta(hours=hours_ago)
                online = _RNG.random() > 0.04
                rows.append(Row(camera_id=cam_id, store_id=sid, ts=ts, online=online))
    return rows


_cameras = _build_camera_status()
(spark.createDataFrame(_cameras)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.camera_status"))
LOG.info("Wrote %d camera_status rows", len(_cameras))

## Detections

Synthetic bounding-box detections for the last 30 days across all stores.
Class distribution leans heavily on `vehicle`, `person`, and `pizza` to match
the quick-serve restaurant scenario.

In [ ]:
_LABEL_WEIGHTS = {"vehicle": 0.42, "person": 0.28, "truck": 0.10, "package": 0.07, "pizza": 0.13}
_CLASS_IDS = {"vehicle": 2, "person": 0, "truck": 7, "package": 84, "pizza": 53}


def _build_detections() -> list[Row]:
    rows: list[Row] = []
    det_id = 0
    for days_ago in range(0, 30):
        per_day = _RNG.randint(220, 360)
        for _ in range(per_day):
            det_id += 1
            sid, *_rest = _RNG.choice(_STORE_DEFS)
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            label = _RNG.choices(list(_LABEL_WEIGHTS), weights=list(_LABEL_WEIGHTS.values()), k=1)[0]
            confidence = round(0.75 + _RNG.random() * 0.24, 3)
            x1 = _RNG.randint(20, 900)
            y1 = _RNG.randint(20, 500)
            x2 = x1 + _RNG.randint(40, 300)
            y2 = y1 + _RNG.randint(40, 220)
            frame_id = f"frame_{det_id:06d}"
            rows.append(Row(
                id=det_id,
                frame_id=frame_id,
                ts=ts,
                store_id=sid,
                label=label,
                class_id=_CLASS_IDS[label],
                confidence=confidence,
                bbox=[x1, y1, x2, y2],
            ))
    return rows


_dets = _build_detections()
(spark.createDataFrame(_dets)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.detections"))
LOG.info("Wrote %d detections", len(_dets))

## License plates

Plate captures over the last 30 days. Plate numbers are partially masked.

In [ ]:
def _build_plates() -> list[Row]:
    rows: list[Row] = []
    plate_id = 0
    for days_ago in range(0, 30):
        per_day = _RNG.randint(80, 160)
        for _ in range(per_day):
            plate_id += 1
            sid, *_rest = _RNG.choice(_STORE_DEFS)
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            state = _RNG.choices(_STATES, weights=_STATE_WEIGHTS, k=1)[0]
            confidence = round(0.90 + _RNG.random() * 0.09, 3)
            prefix = "".join(_RNG.choice("ABCDEFGHJKLMNPRSTUVWXYZ") for _ in range(3))
            rows.append(Row(
                id=plate_id,
                ts=ts,
                store_id=sid,
                state=state,
                plate_masked=f"{prefix}***",
                confidence=confidence,
            ))
    return rows


_plates = _build_plates()
(spark.createDataFrame(_plates)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.license_plates"))
LOG.info("Wrote %d license_plates", len(_plates))

## Inventory

Pizza stock percentage and truck-parking capacity for the first four stores
over the last 12 hours, sampled every 30 minutes.

In [ ]:
import math


def _build_inventory() -> list[Row]:
    rows: list[Row] = []
    for sid, *_ in _STORE_DEFS[:4]:
        for half_hours_ago in range(0, 24):
            ts = _NOW - timedelta(minutes=30 * half_hours_ago)
            hour_pos = (ts.hour + ts.minute / 60.0)
            pizza_pct = 100.0 - max(0.0, 12.0 * math.sin((hour_pos - 6) * math.pi / 12.0)) - _RNG.uniform(0, 8)
            pizza_pct = max(15.0, min(100.0, pizza_pct))
            truck_pct = 30.0 + 30.0 * math.sin((hour_pos - 6) * math.pi / 12.0) + _RNG.uniform(-6, 6)
            truck_pct = max(0.0, min(100.0, truck_pct))
            rows.append(Row(ts=ts, store_id=sid, item="pizza",         percentage=round(pizza_pct, 1)))
            rows.append(Row(ts=ts, store_id=sid, item="truck_parking", percentage=round(truck_pct, 1)))
    return rows


_inv = _build_inventory()
(spark.createDataFrame(_inv)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.inventory"))
LOG.info("Wrote %d inventory rows", len(_inv))

## Alerts (Jolt rules)

Recent rule-engine alert events for the Alerts tab.

In [ ]:
_ALERT_RULES = [
    ("temperature_critical", "Refrigeration temperature > 90F", "critical"),
    ("temperature_warning",  "Refrigeration temperature > 80F", "warning"),
    ("pizza_low_stock",      "Pizza inventory dropped below 25%", "warning"),
    ("camera_offline",       "Camera offline > 5 minutes", "warning"),
    ("vehicle_dwell_long",   "Vehicle dwell time > 8 minutes at drive-through", "info"),
    ("unrecognized_plate",   "Unrecognized license plate at restricted lane", "info"),
]


def _build_alerts() -> list[Row]:
    rows: list[Row] = []
    alert_id = 0
    for days_ago in range(0, 7):
        per_day = _RNG.randint(8, 18)
        for _ in range(per_day):
            alert_id += 1
            sid, store_name, *_ = _RNG.choice(_STORE_DEFS)
            rule_id, message, severity = _RNG.choice(_ALERT_RULES)
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            rows.append(Row(
                id=alert_id,
                ts=ts,
                store_id=sid,
                store_name=store_name,
                rule_id=rule_id,
                message=message,
                severity=severity,
                acknowledged=days_ago >= 1,
            ))
    return rows


_alerts = _build_alerts()
(spark.createDataFrame(_alerts)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.alerts"))
LOG.info("Wrote %d alerts", len(_alerts))

## App write-back mirror tables\n\nThe LensIQ app writes high-frequency demo state (guest counts, plate reads,\nfog observations, spill cycles, face matches) to Lakebase. Genie cannot query\nLakebase directly, so the app also dual-writes to these UC tables via the\n`analytics` plugin. Seed them with a few days of synthetic baseline data so\nthe Genie space returns useful business answers even before any booth demo\nhas been run.

In [ ]:
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

_ZONES = ["forecourt", "drive_thru", "entrance", "register", "aisle"]
_OCR_MODELS = ["databricks-claude-opus-4-7", "lensiq-license-plate"]
_CAMERA_LABELS = ["Forecourt", "Drive-thru", "Entrance", "Aisle", "Register"]
_PLATE_CHARS = "ABCDEFGHJKLMNPRSTUVWXYZ"
_FACES = [
    ("Banned Person A", "banned"),
    ("Banned Person B", "banned"),
    ("Banned Person C", "banned"),
    ("VIP Guest A",     "vip"),
    ("VIP Guest B",     "vip"),
    ("VIP Guest C",     "vip"),
    ("VIP Guest D",     "vip"),
    ("Staff Member A",  "staff"),
    ("Staff Member B",  "staff"),
    ("Staff Member C",  "staff"),
]

_GUEST_SCHEMA = StructType([
    StructField("id",           LongType(),   False),
    StructField("ts",           TimestampType(), False),
    StructField("source_id",    StringType(), False),
    StructField("zone",         StringType(), False),
    StructField("person_count", IntegerType(),False),
    StructField("store_id",     StringType(), True),
])

_PLATE_READ_SCHEMA = StructType([
    StructField("id",                   LongType(),     False),
    StructField("ts",                   TimestampType(),False),
    StructField("source_id",            StringType(),   False),
    StructField("store_id",             StringType(),   True),
    StructField("plate_text",           StringType(),   False),
    StructField("confidence",           DoubleType(),   False),
    StructField("ocr_model",            StringType(),   True),
    StructField("detection_confidence", DoubleType(),   True),
])

_FOG_SCHEMA = StructType([
    StructField("id",           LongType(),     False),
    StructField("ts",           TimestampType(),False),
    StructField("source_id",    StringType(),   False),
    StructField("store_id",     StringType(),   True),
    StructField("camera_label", StringType(),   False),
    StructField("fogged",       BooleanType(),  False),
    StructField("region_count", IntegerType(),  False),
    StructField("area_pct",     DoubleType(),   False),
])

_SPILL_SCHEMA = StructType([
    StructField("id",             LongType(),     False),
    StructField("ts",             TimestampType(),False),
    StructField("source_id",      StringType(),   False),
    StructField("store_id",       StringType(),   True),
    StructField("spill_first_ts", TimestampType(),False),
    StructField("cone_first_ts",  TimestampType(),False),
    StructField("response_ms",    IntegerType(),  False),
    StructField("was_assisted",   BooleanType(),  False),
])

_FACE_MATCH_SCHEMA = StructType([
    StructField("id",         LongType(),     False),
    StructField("ts",         TimestampType(),False),
    StructField("source_id",  StringType(),   False),
    StructField("store_id",   StringType(),   True),
    StructField("face_id",    LongType(),     True),
    StructField("name",       StringType(),   False),
    StructField("role",       StringType(),   False),
    StructField("similarity", DoubleType(),   False),
])


def _random_store() -> tuple[str, str]:
    sid, name, *_ = _RNG.choice(_STORE_DEFS)
    return sid, name


def _build_guest_counts() -> list[Row]:
    rows: list[Row] = []
    next_id = 1
    for days_ago in range(0, 7):
        per_day = _RNG.randint(180, 260)
        for _ in range(per_day):
            sid, _name = _random_store()
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            zone = _RNG.choice(_ZONES)
            # Forecourt typically has more activity than aisles.
            if zone == "forecourt":
                cnt = _RNG.randint(1, 6)
            elif zone == "drive_thru":
                cnt = _RNG.randint(0, 4)
            else:
                cnt = _RNG.randint(0, 3)
            rows.append(Row(
                id=next_id,
                ts=ts,
                source_id=f"cam-{zone}-{_RNG.randint(1, 3)}",
                zone=zone,
                person_count=cnt,
                store_id=sid,
            ))
            next_id += 1
    return rows


def _build_plate_reads() -> list[Row]:
    rows: list[Row] = []
    next_id = 1
    for days_ago in range(0, 7):
        per_day = _RNG.randint(60, 140)
        for _ in range(per_day):
            sid, _name = _random_store()
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            length = _RNG.choice([6, 7])
            plate_text = "".join(
                _RNG.choice("0123456789" if _RNG.random() < 0.4 else _PLATE_CHARS)
                for _ in range(length)
            )
            rows.append(Row(
                id=next_id,
                ts=ts,
                source_id=f"cam-drivethru-{_RNG.randint(1, 2)}",
                store_id=sid,
                plate_text=plate_text,
                confidence=round(0.78 + _RNG.random() * 0.20, 3),
                ocr_model=_RNG.choice(_OCR_MODELS),
                detection_confidence=round(0.65 + _RNG.random() * 0.30, 3),
            ))
            next_id += 1
    return rows


def _build_fog_observations() -> list[Row]:
    rows: list[Row] = []
    next_id = 1
    # 24 hours of ticks per store-camera pair, every 5 minutes.
    for sid, *_rest in _STORE_DEFS:
        for cam_idx, cam_label in enumerate(_CAMERA_LABELS[:3]):
            # Each camera has a different baseline foggedness so analytics
            # surface real differences instead of uniform noise.
            base_area = _RNG.uniform(2.0, 18.0)
            for minutes_ago in range(0, 24 * 60, 5):
                ts = _NOW - timedelta(minutes=minutes_ago)
                area = max(0.0, min(100.0, base_area + _RNG.uniform(-3.0, 6.0)))
                fogged = area >= 25.0
                rows.append(Row(
                    id=next_id,
                    ts=ts,
                    source_id=f"{sid}-CAM-{cam_idx+1:02d}",
                    store_id=sid,
                    camera_label=cam_label,
                    fogged=fogged,
                    region_count=int(area // 10),
                    area_pct=round(area, 2),
                ))
                next_id += 1
    return rows


def _build_spill_cycles() -> list[Row]:
    rows: list[Row] = []
    next_id = 1
    for days_ago in range(0, 7):
        per_day = _RNG.randint(2, 6)
        for _ in range(per_day):
            sid, _name = _random_store()
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            # Response time is log-normal-ish: most cycles are 20-90s, a
            # few outliers in the 3-6 minute range.
            if _RNG.random() < 0.85:
                response_ms = _RNG.randint(20_000, 90_000)
            else:
                response_ms = _RNG.randint(180_000, 360_000)
            spill_first = ts - timedelta(milliseconds=response_ms)
            rows.append(Row(
                id=next_id,
                ts=ts,
                source_id=f"cam-aisle-{_RNG.randint(1, 3)}",
                store_id=sid,
                spill_first_ts=spill_first,
                cone_first_ts=ts,
                response_ms=response_ms,
                was_assisted=_RNG.random() < 0.4,
            ))
            next_id += 1
    return rows


def _build_face_matches() -> list[Row]:
    rows: list[Row] = []
    next_id = 1
    for days_ago in range(0, 7):
        per_day = _RNG.randint(8, 22)
        for _ in range(per_day):
            sid, _name = _random_store()
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            face_idx = _RNG.randint(0, len(_FACES) - 1)
            name, role = _FACES[face_idx]
            rows.append(Row(
                id=next_id,
                ts=ts,
                source_id=f"cam-entrance-{_RNG.randint(1, 2)}",
                store_id=sid,
                face_id=face_idx + 1,
                name=name,
                role=role,
                similarity=round(0.50 + _RNG.random() * 0.45, 3),
            ))
            next_id += 1
    return rows


def _write_mirror(rows: list[Row], schema: StructType, table: str) -> int:
    df = spark.createDataFrame(rows, schema=schema) if rows else spark.createDataFrame([], schema)
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{FQN}.{table}"))
    return df.count()


for tbl, rows, schema in [
    ("guest_counts",     _build_guest_counts(),     _GUEST_SCHEMA),
    ("plate_reads",      _build_plate_reads(),      _PLATE_READ_SCHEMA),
    ("fog_observations", _build_fog_observations(), _FOG_SCHEMA),
    ("spill_cycles",     _build_spill_cycles(),     _SPILL_SCHEMA),
    ("face_matches",     _build_face_matches(),     _FACE_MATCH_SCHEMA),
]:
    cnt = _write_mirror(rows, schema, tbl)
    LOG.info("Wrote %d %s rows", cnt, tbl)

## Table + column comments\n\nApply Unity Catalog table and column COMMENTs so Genie sees the business\ncontext for every field without needing to repeat it in the Genie space JSON.\n`ALTER TABLE ... SET TBLPROPERTIES` is idempotent; column comments are set\nvia `ALTER TABLE ... ALTER COLUMN ... COMMENT`.

In [ ]:
_TABLE_COMMENTS: dict[str, str] = {
    "stores": "The 8 fixed LensIQ store locations. Join target for every other table - always surface stores.name in answers.",
    "devices": "One refrigeration / IoT device per store. Current temperature and status snapshot.",
    "device_readings": "Hourly time series of temperature + humidity readings per device for the last 7 days. >80F is warning, >90F is critical.",
    "camera_status": "Per-hour online/offline samples for every camera in the fleet. Each store has 3 cameras.",
    "detections": "Per-frame YOLO detections from every camera. label is one of vehicle | truck | person | pizza | package. Confidence is 0-1.",
    "license_plates": "Drive-thru / forecourt license plate captures. Plate text is partially masked for privacy; pair with plate_reads for the live full-text view.",
    "inventory": "Pizza hot-hold stock and truck-parking fill samples every 30 minutes for the first 4 stores. item is either 'pizza' (% stocked) or 'truck_parking' (% occupied).",
    "alerts": "Rule-engine alerts emitted by the LensIQ Jolt subsystem. Severity is critical | warning | info.",
    "guest_counts": "Live write-back from the LensIQ Guests page. Each row is one person-count sample for a defined zone. Lands within ~2s of the booth click.",
    "plate_reads": "Live write-back from the LensIQ License Plates page. One row per successful OCR with full plate text and OCR confidence.",
    "fog_observations": "Live write-back from the LensIQ Camera Health page. Per-tick lens-condition observation. area_pct >= 25 is treated as fogged.",
    "spill_cycles": "Live write-back from the LensIQ Spills page. One row per completed spill -> cone response cycle. response_ms target is < 60000.",
    "face_matches": "Live write-back from the LensIQ Facial Recognition page. One row per matched face above cosine-similarity 0.45. role is banned | vip | staff.",
}

_COLUMN_COMMENTS: dict[str, dict[str, str]] = {
    "stores": {
        "id": "Store identifier (e.g. S-ATL-001). Foreign key for store_id in every other table.",
        "name": "Human-readable store name. Use this in answers, never the raw id.",
        "location": "City, state of the store.",
        "lat": "Latitude in decimal degrees.",
        "lng": "Longitude in decimal degrees.",
    },
    "detections": {
        "id": "Unique detection id.",
        "frame_id": "Source video frame the detection came from.",
        "ts": "UTC timestamp.",
        "store_id": "Store where the detection was captured. Joins to stores.id.",
        "label": "Class label: vehicle | truck | person | pizza | package.",
        "class_id": "COCO class id (2=vehicle, 0=person, 7=truck, 84=package, 53=pizza).",
        "confidence": "Model confidence 0.0 to 1.0. Filter < 0.5 for executive answers unless user asks for it.",
        "bbox": "Bounding box [x1, y1, x2, y2] integer pixels in the source frame.",
    },
    "license_plates": {
        "id": "Unique capture id.",
        "ts": "UTC timestamp of the capture.",
        "store_id": "Store where the plate was seen.",
        "state": "Two-letter US state abbreviation extracted from the plate.",
        "plate_masked": "Plate text with final characters redacted (e.g. ABC***) for privacy.",
        "confidence": "OCR confidence 0.0 to 1.0.",
    },
    "alerts": {
        "id": "Unique alert id.",
        "ts": "UTC timestamp the rule fired.",
        "store_id": "Store associated with the alert.",
        "store_name": "Denormalized display name (matches stores.name).",
        "rule_id": "Rule identifier (e.g. temperature_critical, pizza_low_stock, camera_offline, vehicle_dwell_long).",
        "message": "Human-readable alert text.",
        "severity": "critical | warning | info. Treat critical as P1.",
        "acknowledged": "Whether an operator has acknowledged the alert.",
    },
    "camera_status": {
        "camera_id": "Camera identifier (<store_id>-CAM-<NN>).",
        "store_id": "Store the camera belongs to.",
        "ts": "UTC hour stamp.",
        "online": "true when the camera was reachable in that hour.",
    },
    "devices": {
        "id": "Device identifier.",
        "name": "Friendly device label.",
        "location": "City, state of the device.",
        "current_temp": "Latest temperature reading in Fahrenheit.",
        "status": "normal (<80F) | warning (80-90F) | critical (>90F).",
        "last_update": "When the device last reported.",
    },
    "device_readings": {
        "device_id": "Joins to devices.id.",
        "ts": "UTC timestamp of the reading.",
        "temperature": "Reading in Fahrenheit. >80F is warning, >90F is critical.",
        "humidity": "Relative humidity 0-100.",
        "status": "Threshold-derived status at this reading.",
    },
    "inventory": {
        "ts": "UTC timestamp.",
        "store_id": "Store the sample was taken at.",
        "item": "Either 'pizza' or 'truck_parking'.",
        "percentage": "0-100 percent. For pizza: stock remaining (lower = problem). For truck_parking: lot fill.",
    },
    "guest_counts": {
        "id": "Unique sample id.",
        "ts": "UTC timestamp.",
        "source_id": "Camera or video feed source the count came from.",
        "zone": "Zone label (entrance, forecourt, drive_thru, register, aisle).",
        "person_count": "Number of distinct persons detected in the zone at that tick.",
        "store_id": "Store the count applies to. Joins to stores.id.",
    },
    "plate_reads": {
        "id": "Unique read id.",
        "ts": "UTC timestamp.",
        "source_id": "Camera / video feed the plate was captured from.",
        "store_id": "Store the read happened at.",
        "plate_text": "Full uppercase alphanumeric plate text, no punctuation.",
        "confidence": "OCR confidence 0.0 to 1.0.",
        "ocr_model": "Serving endpoint that produced the OCR.",
        "detection_confidence": "Vehicle-detection confidence for the source frame.",
    },
    "fog_observations": {
        "id": "Unique observation id.",
        "ts": "UTC timestamp.",
        "source_id": "Camera / feed source id.",
        "store_id": "Store the camera belongs to.",
        "camera_label": "Friendly camera label (Forecourt, Drive-thru, Entrance, etc.).",
        "fogged": "true when area_pct crossed the fog threshold for this tick.",
        "region_count": "Number of distinct fog regions detected in the frame.",
        "area_pct": "Percent of frame fogged (0-100). >= 25 is concerning.",
    },
    "spill_cycles": {
        "id": "Unique cycle id.",
        "ts": "When the cycle completed (cone went down).",
        "source_id": "Camera / feed source.",
        "store_id": "Store the cycle happened at.",
        "spill_first_ts": "Timestamp of first spill detection.",
        "cone_first_ts": "Timestamp of first cone detection.",
        "response_ms": "Spill -> cone response time in milliseconds. Target is < 60000 (1 minute).",
        "was_assisted": "true when an operator was prompted by the system.",
    },
    "face_matches": {
        "id": "Unique match id.",
        "ts": "UTC timestamp.",
        "source_id": "Camera / feed source.",
        "store_id": "Store the match happened at.",
        "face_id": "Enrolled face id this match was attached to.",
        "name": "Person's enrolled name.",
        "role": "banned | vip | staff.",
        "similarity": "Cosine similarity 0.0 to 1.0. Higher is a stronger match.",
    },
}


def _sql_escape(text: str) -> str:
    return text.replace("'", "''")


for table, comment in _TABLE_COMMENTS.items():
    spark.sql(
        f"COMMENT ON TABLE {FQN}.{table} IS '{_sql_escape(comment)}'"
    )
    cols = _COLUMN_COMMENTS.get(table, {})
    for col, col_comment in cols.items():
        # ALTER TABLE ... ALTER COLUMN ... COMMENT is supported by Delta on UC.
        # Skip silently if the column was dropped from the schema in a later
        # edit so this seed stays forward-compatible.
        try:
            spark.sql(
                f"ALTER TABLE {FQN}.{table} ALTER COLUMN {col} "
                f"COMMENT '{_sql_escape(col_comment)}'"
            )
        except Exception as exc:
            LOG.warning("skip column comment %s.%s: %s", table, col, exc)
    LOG.info("Comments applied: %s.%s (%d columns)", FQN, table, len(cols))

In [ ]:
tables = [
    "stores", "devices", "device_readings", "camera_status",
    "detections", "license_plates", "inventory", "alerts",
    "guest_counts", "plate_reads", "fog_observations",
    "spill_cycles", "face_matches",
]
for tbl in tables:
    cnt = spark.table(f"{FQN}.{tbl}").count()
    LOG.info("%s.%s  rows=%d", FQN, tbl, cnt)